# Canada Economic Forecaster — Modelling Notebook

**Goal:** Forecast wages, shelter CPI, and employment using a Linear Regression baseline vs. Random Forest.

**Data:** Statistics Canada processed CSVs (2014–2024)  
**Targets:** `avg_hourly_wage` · `shelter_index` · `avg_employed_thousands`  
**Horizon:** 2025–2027 via autoregressive rollout

In [ ]:
import sys, os
sys.path.insert(0, os.path.join('..', 'src'))

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

DATA_DIR = os.path.join('..', 'data')
print('Setup complete')

## 1. Feature Engineering

In [ ]:
from build_features import load_raw, make_wage_features, make_shelter_features, make_employment_features

wages_cpi, employment = load_raw()

wage_df    = make_wage_features(wages_cpi)
shelter_df = make_shelter_features(wages_cpi)
emp_df     = make_employment_features(employment)

print(f'Wage features:      {wage_df.shape}')
print(f'Shelter features:   {shelter_df.shape}')
print(f'Employment features:{emp_df.shape}')
wage_df.tail()

## 2. EDA — Historical trends

In [ ]:
# National wage vs shelter index
nat = wages_cpi[wages_cpi['geo'] == 'Canada'].sort_values('year')

fig = go.Figure()
fig.add_trace(go.Scatter(x=nat['year'], y=nat['wage_index'], name='Wage Index',
                         line=dict(color='#3b82f6', width=3)))
fig.add_trace(go.Scatter(x=nat['year'], y=nat['shelter_index'], name='Shelter CPI',
                         line=dict(color='#f43f5e', width=3, dash='dash')))
fig.add_hline(y=100, line_color='grey', line_dash='dot', opacity=0.5)
fig.update_layout(title='Canada — Wage vs Shelter Index (2014=100)',
                  xaxis_title='Year', yaxis_title='Index', template='plotly_dark')
fig.show()

In [ ]:
# Provincial avg hourly wage distribution
prov = wages_cpi[wages_cpi['geo'] != 'Canada']
fig2 = px.line(prov, x='year', y='avg_hourly_wage', color='geo',
               title='Average Hourly Wage by Province ($/hr)',
               template='plotly_dark')
fig2.show()

## 3. Model Training

Train/test split: **train < 2022**, **test ≥ 2022**

In [ ]:
from train_models import (
    train_and_forecast,
    WAGE_FEATURES, SHELTER_FEATURES, EMPLOYMENT_FEATURES,
)

# Save feature files first
wage_df.to_csv(os.path.join(DATA_DIR, 'features_wages.csv'), index=False)
shelter_df.to_csv(os.path.join(DATA_DIR, 'features_shelter.csv'), index=False)
emp_df.to_csv(os.path.join(DATA_DIR, 'features_employment.csv'), index=False)

metrics_wages    = train_and_forecast('wages',      'features_wages.csv',
                                       WAGE_FEATURES, 'avg_hourly_wage', 'geo')
metrics_shelter  = train_and_forecast('shelter',    'features_shelter.csv',
                                       SHELTER_FEATURES, 'shelter_index', 'geo')
metrics_emp      = train_and_forecast('employment', 'features_employment.csv',
                                       EMPLOYMENT_FEATURES, 'avg_employed_thousands', 'industry')

## 4. Model Comparison

In [ ]:
rows = []
for name, m in [('Wages', metrics_wages), ('Shelter CPI', metrics_shelter), ('Employment', metrics_emp)]:
    rows.append({'Target': name, 'Model': 'LinearRegression', **m['linear']})
    rows.append({'Target': name, 'Model': 'RandomForest',     **m['random_forest']})

comparison = pd.DataFrame(rows).set_index(['Target', 'Model'])
comparison.style.format('{:.4f}').highlight_min(subset=['MAE', 'RMSE'], color='#1a4d2e')\
                                  .highlight_max(subset=['R2'], color='#1a4d2e')

## 5. Forecast Visualization

In [ ]:
fc_wages = pd.read_csv(os.path.join(DATA_DIR, 'forecasts_wages.csv'))

province = 'Ontario'
hist = wages_cpi[wages_cpi['geo'] == province].sort_values('year')
fc   = fc_wages[fc_wages['geo'] == province]

bridge_x  = [hist['year'].iloc[-1]] + fc['year'].tolist()
bridge_rf = [hist['avg_hourly_wage'].iloc[-1]] + fc['rf_forecast'].tolist()
bridge_lo = [hist['avg_hourly_wage'].iloc[-1]] + fc['rf_lower'].tolist()
bridge_hi = [hist['avg_hourly_wage'].iloc[-1]] + fc['rf_upper'].tolist()
bridge_lr = [hist['avg_hourly_wage'].iloc[-1]] + fc['lr_forecast'].tolist()

fig3 = go.Figure()
fig3.add_trace(go.Scatter(x=hist['year'], y=hist['avg_hourly_wage'],
                           name='Historical', line=dict(color='#3b82f6', width=3)))
fig3.add_trace(go.Scatter(
    x=bridge_x + bridge_x[::-1],
    y=bridge_hi + bridge_lo[::-1],
    fill='toself', fillcolor='rgba(34,197,94,0.15)',
    line=dict(color='rgba(0,0,0,0)'), name='RF 90% PI',
))
fig3.add_trace(go.Scatter(x=bridge_x, y=bridge_rf, name='RF Forecast',
                           line=dict(color='#22c55e', width=2.5, dash='dot')))
fig3.add_trace(go.Scatter(x=bridge_x, y=bridge_lr, name='LR Baseline',
                           line=dict(color='#f59e0b', width=2, dash='dash')))
fig3.add_vline(x=hist['year'].iloc[-1] + 0.5, line_dash='dash', line_color='grey')
fig3.update_layout(title=f'{province} — Avg Hourly Wage Forecast ($/hr)',
                    template='plotly_dark', xaxis_title='Year', yaxis_title='$/hr')
fig3.show()

## 6. Feature Importance

In [ ]:
LABELS = {
    'avg_hourly_wage_lag1': 'Wage (t−1)', 'avg_hourly_wage_lag2': 'Wage (t−2)',
    'avg_hourly_wage_yoy': 'Wage YoY %', 'shelter_index_lag1': 'Shelter CPI (t−1)',
    'cpi_index_lag1': 'All-Items CPI (t−1)', 'province_code': 'Province',
    'year': 'Year (trend)', 'covid': 'COVID dummy',
}

imp = pd.read_csv(os.path.join(DATA_DIR, 'importance_wages.csv'))
imp['label'] = imp['feature'].map(LABELS).fillna(imp['feature'])
imp = imp.sort_values('importance')

fig4 = go.Figure(go.Bar(
    x=imp['importance'], y=imp['label'],
    orientation='h', marker_color='#3b82f6',
    text=imp['importance'].round(3), textposition='outside',
))
fig4.update_layout(title='RF Feature Importance — Wage Model',
                    template='plotly_dark',
                    xaxis_title='Importance (mean decrease in impurity)')
fig4.show()

## 7. Provincial Forecast Comparison — 2027

In [ ]:
fc_shelter = pd.read_csv(os.path.join(DATA_DIR, 'forecasts_shelter.csv'))
fc_w       = pd.read_csv(os.path.join(DATA_DIR, 'forecasts_wages.csv'))

prov_2027 = fc_shelter[fc_shelter['year'] == 2027][['geo', 'rf_forecast']].rename(
    columns={'rf_forecast': 'shelter_2027'})
wage_2027 = fc_w[fc_w['year'] == 2027][['geo', 'rf_forecast']].rename(
    columns={'rf_forecast': 'wage_2027'})

combined = prov_2027.merge(wage_2027, on='geo')
combined['gap_2027'] = combined['shelter_2027'] - combined['wage_2027']
combined = combined.sort_values('gap_2027', ascending=True)

fig5 = go.Figure(go.Bar(
    x=combined['gap_2027'], y=combined['geo'],
    orientation='h',
    marker_color=combined['gap_2027'].apply(lambda x: '#f43f5e' if x > 0 else '#22c55e'),
    text=combined['gap_2027'].round(1), textposition='outside',
))
fig5.add_vline(x=0, line_color='grey', line_width=1.5)
fig5.update_layout(
    title='Projected Affordability Gap by Province — 2027 (Shelter Index − Wage Index)',
    xaxis_title='Index points (red = shelter outpaces wages)',
    template='plotly_dark',
)
fig5.show()